# 🎯 Training and Fine-Tuning Open-Source Models
### Google Colab Edition — GPU Accelerated

**What this notebook covers:**
1. 📖 Key Concepts — Fine-Tuning, LoRA, PEFT (explained by LLaMA via Groq)
2. 📊 Dataset Preparation — instruction-response pairs
3. 🏋️ Fine-Tune `TinyLlama-1.1B` with LoRA on Colab GPU
4. ✍️ Generate text before vs after fine-tuning
5. 🤖 Groq API — instant answers without any training

---

### ⚙️ Before You Start — Enable GPU in Colab
1. Click **Runtime** → **Change runtime type**
2. Set **Hardware accelerator** → **T4 GPU**
3. Click **Save**
4. Then run cells from top to bottom

### 🔑 You Need
- A **free Groq API key** from https://console.groq.com (for Sections 1 & 5)
- That's it — everything else runs locally on Colab's free GPU!

---

### 🤔 Why TinyLlama instead of GPT-2?

| Model | Params | GPU RAM | Quality | Colab T4 |
|-------|--------|---------|---------|----------|
| GPT-2 | 117M | ~1GB | Basic | ✅ |
| **TinyLlama-1.1B** | **1.1B** | **~3GB** | **Much better** | **✅** |
| LLaMA-3.2 3B | 3B | ~8GB | Great | ✅ (tight) |
| LLaMA-3.1 8B | 8B | ~16GB | Excellent | ❌ |

**TinyLlama-1.1B** is the sweet spot for Colab — modern LLaMA architecture, much better than GPT-2, fits easily on T4 GPU with LoRA.

---
## 🔧 Cell 1 — Install All Required Libraries

In [ ]:
# Install all required packages
# This takes about 2-3 minutes on first run

!pip install -q \
    transformers \
    peft \
    accelerate \
    bitsandbytes \
    langchain-groq \
    langchain-core \
    groq \
    datasets

print("✅ All packages installed!")

---
## 🔑 Cell 2 — Set Your Groq API Key

In [ ]:
import os

# ── Option A: Paste your key directly (easiest for students) ──────────────────
GROQ_API_KEY = "your_groq_api_key_here"   # ← paste your key here

# ── Option B: Use Colab Secrets (more secure) ─────────────────────────────────
# Click the 🔑 key icon in the left sidebar → Add secret → Name: GROQ_API_KEY
# Then uncomment these lines:
# from google.colab import userdata
# GROQ_API_KEY = userdata.get('GROQ_API_KEY')

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

if GROQ_API_KEY and GROQ_API_KEY != "your_groq_api_key_here":
    print("✅ GROQ_API_KEY set!")
    print(f"   Key preview: {GROQ_API_KEY[:8]}...")
else:
    print("⚠️  Please paste your Groq API key above.")
    print("   Get a FREE key at: https://console.groq.com")

---
## 🖥️ Cell 3 — Check GPU and Import Libraries

In [ ]:
import os
import time
import json
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import (
    get_peft_model,
    LoraConfig,
    TaskType,
    PeftModel,
)
from datasets import Dataset
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from groq import Groq

# ── Check GPU ─────────────────────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("=" * 55)
print("  ENVIRONMENT STATUS")
print("=" * 55)
print(f"  Device           : {DEVICE.upper()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  GPU              : {gpu_name}")
    print(f"  GPU Memory       : {gpu_mem:.1f} GB")
    print(f"  PyTorch CUDA     : {torch.version.cuda}")
else:
    print("  ⚠️  No GPU found!")
    print("  → Go to Runtime → Change runtime type → T4 GPU")

print(f"  PyTorch version  : {torch.__version__}")
print(f"  Groq API Key     : {'✅ Set' if os.environ.get('GROQ_API_KEY') else '❌ Missing'}")
print("=" * 55)

---
## 📖 Section 1 — Key Concepts (Explained by LLaMA via Groq)

This section uses the **Groq API** to have LLaMA explain fine-tuning concepts in simple terms.

In [ ]:
# ── Concept explainer using Groq LLaMA 3.1 ───────────────────────────────────

def explain_concept(concept: str, model: str = "llama-3.1-8b-instant") -> str:
    """
    Use LangChain + Groq to explain an AI/ML concept simply.
    LLaMA 3.1 8B via Groq — fast and free.
    """
    llm = ChatGroq(
        api_key=os.environ.get("GROQ_API_KEY"),
        model_name=model,
        temperature=0.3,
        max_tokens=400,
    )
    chain = (
        ChatPromptTemplate.from_messages([
            ("system",
             "You are an AI tutor. Explain concepts simply in 3-4 sentences. "
             "Use a simple real-world analogy. Avoid jargon."),
            ("human", "Explain this concept clearly: {concept}"),
        ])
        | llm
        | StrOutputParser()
    )
    return chain.invoke({"concept": concept})


# ── Fine-Tuning vs Other Approaches — reference table ────────────────────────
print("📊 Fine-Tuning vs Other Approaches")
print("=" * 75)
print(f"{'Approach':<30} {'When to use':<25} {'Speed':<12} {'Cost'}")
print("-" * 75)
approaches = [
    ("Pre-trained model as-is",  "General tasks",          "Instant",      "Free"),
    ("Prompt engineering",        "Custom behaviour",       "Instant",      "API cost"),
    ("LoRA fine-tuning",          "Domain-specific tasks",  "Mins-hours",   "Free (local)"),
    ("Full fine-tuning",          "Complete customisation", "Days",         "High (GPU)"),
]
for a in approaches:
    print(f"  {a[0]:<28} {a[1]:<25} {a[2]:<12} {a[3]}")
print("=" * 75)

In [ ]:
# ── Ask LLaMA to explain any concept ─────────────────────────────────────────
# Change the concept below to learn about different topics

CONCEPTS = {
    1: "Fine-tuning a pre-trained language model",
    2: "Parameter-Efficient Fine-Tuning (PEFT)",
    3: "LoRA: Low-Rank Adaptation for LLMs",
    4: "The HuggingFace Trainer class",
    5: "Preparing datasets for fine-tuning LLMs",
    6: "When should you fine-tune vs use a pre-trained model?",
    7: "What is TinyLlama and why is it good for fine-tuning?",
    8: "What is the difference between LoRA and QLoRA?",
}

print("Available concepts:")
for k, v in CONCEPTS.items():
    print(f"  {k}. {v}")

# ── Change this number (1-8) to pick a concept ───────────────────────────────
SELECTED = 3   # ← change this

print(f"\n🧠 Explaining: '{CONCEPTS[SELECTED]}'")
print("-" * 60)
explanation = explain_concept(CONCEPTS[SELECTED])
print(explanation)

In [ ]:
# ── Explain ALL concepts at once ──────────────────────────────────────────────
# Run this cell to get a full concept guide

core_concepts = [
    "Fine-tuning a pre-trained language model",
    "LoRA: Low-Rank Adaptation for LLMs",
    "What is TinyLlama and why is it good for fine-tuning on Colab GPU?",
]

for concept in core_concepts:
    print(f"\n{'='*60}")
    print(f"📖 {concept}")
    print('='*60)
    print(explain_concept(concept))

---
## 📊 Section 2 — Dataset Preparation

Fine-tuning needs **instruction → response** pairs.
Add your own examples in the list below — 5 to 20 is enough for a demo.

In [ ]:
# ── Training examples (instruction → response pairs) ─────────────────────────
# Edit these or add your own below!

TRAINING_EXAMPLES = [
    {
        "instruction": "What is machine learning?",
        "response": "Machine learning is teaching computers to learn from data without being explicitly programmed."
    },
    {
        "instruction": "What is a neural network?",
        "response": "A neural network is a system of layers that learns patterns by adjusting weights during training."
    },
    {
        "instruction": "What is fine-tuning?",
        "response": "Fine-tuning is taking a pre-trained model and training it further on your specific data."
    },
    {
        "instruction": "What is PEFT?",
        "response": "PEFT stands for Parameter-Efficient Fine-Tuning. It trains only a small subset of parameters, saving time and memory."
    },
    {
        "instruction": "What is LoRA?",
        "response": "LoRA adds small trainable layers to a frozen model. It trains 99% fewer parameters than full fine-tuning."
    },
    {
        "instruction": "What is TinyLlama?",
        "response": "TinyLlama is a compact 1.1B parameter model with the same LLaMA architecture, designed to run efficiently on limited hardware."
    },
    {
        "instruction": "What is a transformer model?",
        "response": "A transformer is a deep learning architecture that uses attention mechanisms to process sequences of data in parallel."
    },
    {
        "instruction": "What is overfitting?",
        "response": "Overfitting happens when a model learns the training data too well and performs poorly on new, unseen data."
    },
    {
        "instruction": "What is a learning rate?",
        "response": "A learning rate controls how much the model updates its weights at each training step. Too high causes instability, too low is very slow."
    },
    {
        "instruction": "What is tokenization?",
        "response": "Tokenization splits text into smaller units called tokens that the model can process. Words, subwords, or characters can be tokens."
    },
]

# ── Add your OWN examples here ────────────────────────────────────────────────
# TRAINING_EXAMPLES.append({
#     "instruction": "Your question here?",
#     "response": "Your answer here."
# })

print(f"✅ Dataset ready: {len(TRAINING_EXAMPLES)} training examples")
print("\nFirst 3 examples:")
print("-" * 60)
for ex in TRAINING_EXAMPLES[:3]:
    print(f"  Q: {ex['instruction']}")
    print(f"  A: {ex['response']}")
    print()

In [ ]:
# ── Format dataset for TinyLlama fine-tuning ──────────────────────────────────
# TinyLlama uses a specific chat template format

def format_example(example: dict) -> str:
    """
    Format a single instruction-response pair into TinyLlama's chat template.

    TinyLlama was trained with this exact format:
    <|system|>\n{system}\n<|user|>\n{instruction}\n<|assistant|>\n{response}

    Using the correct format helps the model learn faster.
    """
    return (
        "<|system|>\n"
        "You are a helpful AI assistant that explains concepts clearly.\n"
        "<|user|>\n"
        f"{example['instruction']}\n"
        "<|assistant|>\n"
        f"{example['response']}"
    )


# Format all examples
formatted_texts = [format_example(ex) for ex in TRAINING_EXAMPLES]

# Create HuggingFace Dataset object
hf_dataset = Dataset.from_dict({"text": formatted_texts})

print(f"✅ Dataset formatted: {len(hf_dataset)} examples")
print("\nFormatted example (how TinyLlama will see it):")
print("=" * 60)
print(formatted_texts[0])
print("=" * 60)

---
## 🏋️ Section 3 — Load TinyLlama + Apply LoRA

**TinyLlama-1.1B-Chat** is a modern LLaMA-architecture model:
- 1.1 billion parameters (10x larger than GPT-2)
- Same architecture as LLaMA-2
- Trained on 3 trillion tokens
- Fits on Colab T4 GPU with LoRA
- Downloads ~2.2GB once, then cached

In [ ]:
# ── Load TinyLlama tokenizer and model ────────────────────────────────────────
# First run downloads ~2.2GB — takes 2-4 minutes
# Subsequent runs load from Colab cache — fast!

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print(f"Loading tokenizer: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"   # important for training stability
print("✅ Tokenizer loaded")

print(f"\nLoading model: {MODEL_ID}")
print("(Downloads ~2.2GB on first run — please wait...)")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,      # float16 saves GPU memory on T4
    device_map="auto",              # automatically puts model on GPU
)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"\n✅ Model loaded!")
print(f"   Model            : {MODEL_ID}")
print(f"   Total parameters : {total_params:,} ({total_params/1e9:.2f}B)")
print(f"   Device           : {next(model.parameters()).device}")
print(f"   dtype            : {next(model.parameters()).dtype}")

if torch.cuda.is_available():
    used_mem = torch.cuda.memory_allocated() / 1e9
    print(f"   GPU memory used  : {used_mem:.2f} GB")

In [ ]:
# ── Generate text BEFORE fine-tuning (baseline) ───────────────────────────────
# Save this output to compare with after fine-tuning!

def generate_response(model, tokenizer, instruction: str, max_new_tokens: int = 100) -> str:
    """
    Generate a response from the model given an instruction.
    Uses the same TinyLlama chat template format.
    """
    prompt = (
        "<|system|>\n"
        "You are a helpful AI assistant that explains concepts clearly.\n"
        "<|user|>\n"
        f"{instruction}\n"
        "<|assistant|>\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens (not the input prompt)
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


# Test with a question from our training data
TEST_QUESTION = "What is LoRA?"

print("📝 BEFORE Fine-Tuning (Base Model)")
print(f"Q: {TEST_QUESTION}")
print("-" * 60)
before_answer = generate_response(model, tokenizer, TEST_QUESTION)
print(f"A: {before_answer}")
print("\n⚠️ Save this output — compare with AFTER fine-tuning!")

In [ ]:
# ── Apply LoRA adapters to TinyLlama ─────────────────────────────────────────
#
# LoRA adds tiny trainable matrices to the attention layers.
# The original model weights stay FROZEN — only LoRA weights train.
# This reduces trainable parameters from 1.1B to ~2M (99.8% reduction!)
#
# Key settings:
#   r=8           → rank of LoRA matrices (higher = more expressive, more memory)
#   lora_alpha=16 → scaling factor (usually 2x rank)
#   target_modules → which layers to add LoRA to (attention layers)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,                           # LoRA rank — 8 is good for Colab T4
    lora_alpha=16,                 # scaling = lora_alpha / r = 2.0
    target_modules=[               # TinyLlama attention layer names
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
    lora_dropout=0.05,
    bias="none",
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)

# Count parameters after LoRA
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params    = total_params - trainable_params
pct_trainable    = trainable_params / total_params * 100

print("✅ LoRA applied to TinyLlama!")
print(f"   Total parameters     : {total_params:>15,}")
print(f"   Frozen parameters    : {frozen_params:>15,}  (original model — unchanged)")
print(f"   Trainable parameters : {trainable_params:>15,}  (LoRA weights only)")
print(f"   % Trainable          : {pct_trainable:.4f}%")
print(f"\n   → Training {trainable_params/1e6:.1f}M params instead of {total_params/1e9:.1f}B")
print(f"   → That's {total_params//trainable_params}x FEWER parameters to update!")

---
## 🚀 Section 4 — Fine-Tune with HuggingFace Trainer

Using HuggingFace `Trainer` — handles the training loop, logging, and checkpointing automatically.

In [ ]:
# ── Tokenize the dataset ───────────────────────────────────────────────────────

MAX_LENGTH = 256   # max tokens per example (256 is enough for our short examples)

def tokenize_function(examples):
    """Tokenize text and set labels = input_ids (standard causal LM training)."""
    tokenized = tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized


# Apply tokenization
tokenized_dataset = hf_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
)

print(f"✅ Dataset tokenized!")
print(f"   Examples     : {len(tokenized_dataset)}")
print(f"   Max length   : {MAX_LENGTH} tokens")
print(f"   Features     : {tokenized_dataset.features}")
print(f"\n   Sample token count: {sum(1 for t in tokenized_dataset[0]['input_ids'] if t != tokenizer.pad_token_id)} real tokens")

In [ ]:
# ── Configure Training ────────────────────────────────────────────────────────
#
# Settings optimized for Colab T4 GPU (15GB VRAM)
# Training 10 examples for 3 epochs takes ~2-5 minutes

OUTPUT_DIR = "/content/tinyllama-finetuned"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # ── Training duration ────────────────────────────────────────────────────
    num_train_epochs=3,            # 3 passes through all examples
    per_device_train_batch_size=1, # 1 example per step (T4 has limited VRAM)
    gradient_accumulation_steps=4, # accumulate 4 steps = effective batch size 4

    # ── Learning rate ────────────────────────────────────────────────────────
    learning_rate=2e-4,            # standard LoRA learning rate
    warmup_ratio=0.1,              # 10% warmup steps
    lr_scheduler_type="cosine",   # cosine decay after warmup

    # ── Memory optimization ──────────────────────────────────────────────────
    fp16=True,                     # float16 training (saves ~50% VRAM)
    optim="adamw_torch",           # AdamW optimizer
    dataloader_pin_memory=False,   # avoid CUDA memory issues

    # ── Logging ──────────────────────────────────────────────────────────────
    logging_steps=1,               # log every step (small dataset)
    logging_dir=f"{OUTPUT_DIR}/logs",
    save_strategy="epoch",         # save after each epoch
    report_to="none",              # no wandb/tensorboard

    # ── Misc ─────────────────────────────────────────────────────────────────
    remove_unused_columns=False,
)

print("✅ Training configuration set!")
print(f"   Epochs           : {training_args.num_train_epochs}")
print(f"   Batch size       : {training_args.per_device_train_batch_size}")
print(f"   Grad accumulation: {training_args.gradient_accumulation_steps}")
print(f"   Effective batch  : {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"   Learning rate    : {training_args.learning_rate}")
print(f"   FP16 training    : {training_args.fp16}")
print(f"   Output directory : {OUTPUT_DIR}")

In [ ]:
# ── Create Trainer and Start Fine-Tuning ─────────────────────────────────────
# This is the main training cell — takes 2-5 minutes on T4 GPU

# Data collator handles padding and batching automatically
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,   # causal LM (not masked LM like BERT)
)

# Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print("🚀 Starting fine-tuning...")
print(f"   Model     : TinyLlama-1.1B-Chat")
print(f"   Examples  : {len(tokenized_dataset)}")
print(f"   Epochs    : {training_args.num_train_epochs}")
print(f"   Device    : {DEVICE.upper()}")
print("="*55)

start_time = time.time()

# ── TRAIN ─────────────────────────────────────────────────────────────────────
train_result = trainer.train()

elapsed = round(time.time() - start_time, 1)

# ── Results ───────────────────────────────────────────────────────────────────
print("\n" + "="*55)
print("✅ FINE-TUNING COMPLETE!")
print("="*55)
print(f"   Training time    : {elapsed}s ({elapsed/60:.1f} mins)")
print(f"   Final loss       : {train_result.training_loss:.4f}")
print(f"   Total steps      : {train_result.global_step}")
print(f"   Trainable params : {trainable_params:,} ({pct_trainable:.4f}%)")

if torch.cuda.is_available():
    peak_mem = torch.cuda.max_memory_allocated() / 1e9
    print(f"   Peak GPU memory  : {peak_mem:.2f} GB")

In [ ]:
# ── Generate text AFTER fine-tuning ───────────────────────────────────────────
# Compare this with the BEFORE output from Section 3!

model.eval()   # switch to evaluation mode

print("📝 AFTER Fine-Tuning (LoRA-adapted Model)")
print(f"Q: {TEST_QUESTION}")
print("-" * 60)
after_answer = generate_response(model, tokenizer, TEST_QUESTION)
print(f"A: {after_answer}")

print("\n" + "="*60)
print("📊 BEFORE vs AFTER Comparison")
print("="*60)
print(f"Question: {TEST_QUESTION}")
print(f"\n🔴 BEFORE: {before_answer}")
print(f"\n🟢 AFTER : {after_answer}")

In [ ]:
# ── Test with multiple questions ──────────────────────────────────────────────

model.eval()

test_questions = [
    "What is machine learning?",
    "What is LoRA?",
    "What is tokenization?",
    "What is overfitting?",
]

print("🧪 Testing fine-tuned model on multiple questions")
print("=" * 60)

for q in test_questions:
    print(f"\nQ: {q}")
    answer = generate_response(model, tokenizer, q, max_new_tokens=80)
    print(f"A: {answer}")
    print("-" * 60)

In [ ]:
# ── Save the fine-tuned LoRA adapter ─────────────────────────────────────────
# Only saves the LoRA weights (~10MB) not the full 2.2GB model

ADAPTER_DIR = "/content/tinyllama-lora-adapter"
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

# Check saved size
import subprocess
result = subprocess.run(["du", "-sh", ADAPTER_DIR], capture_output=True, text=True)

print(f"✅ LoRA adapter saved to: {ADAPTER_DIR}")
print(f"   Saved size: {result.stdout.split()[0]}")
print("   (Only ~10-20MB — just the LoRA weights, not the full 2.2GB model!)")
print("\nTo reload later:")
print("   base_model = AutoModelForCausalLM.from_pretrained('TinyLlama/TinyLlama-1.1B-Chat-v1.0')")
print("   model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)")

---
## 🤖 Section 5 — Groq API (Alternative: No Fine-Tuning Needed)

Instead of fine-tuning, you can use a powerful pre-trained model with a good system prompt.
This approach uses **LangChain + Groq** and gives results in seconds — no GPU needed!

In [ ]:
# ── Groq API Demo — instant answers without training ─────────────────────────

def ask_groq(
    question: str,
    system_prompt: str,
    model: str = "llama-3.3-70b-versatile",
    temperature: float = 0.5,
) -> str:
    """
    Ask a question via Groq API using LLaMA 3.3 70B.
    Returns the answer as a string.
    """
    llm = ChatGroq(
        api_key=os.environ.get("GROQ_API_KEY"),
        model_name=model,
        temperature=temperature,
        max_tokens=512,
    )
    chain = (
        ChatPromptTemplate.from_messages([
            ("system", system_prompt),
            ("human", "{question}"),
        ])
        | llm
        | StrOutputParser()
    )
    return chain.invoke({"question": question})


# ── Customise the system prompt to shape model behaviour ─────────────────────
SYSTEM_PROMPT = (
    "You are an expert AI teacher specialised in explaining machine learning. "
    "Always give short, clear answers with one practical example. "
    "Never use jargon without explaining it first."
)

USER_QUESTION = "What is the difference between fine-tuning and prompt engineering?"

print("🤖 Groq API — LLaMA 3.3 70B")
print(f"Q: {USER_QUESTION}")
print("-" * 60)
answer = ask_groq(USER_QUESTION, SYSTEM_PROMPT)
print(answer)

In [ ]:
# ── Compare different Groq models ─────────────────────────────────────────────

GROQ_MODELS = {
    "LLaMA 3.1 8B  (fast)":     "llama-3.1-8b-instant",
    "LLaMA 3.3 70B (powerful)": "llama-3.3-70b-versatile",
    "Gemma 2 9B":               "gemma2-9b-it",
}

comparison_question = "Explain LoRA fine-tuning in 2 sentences."

print(f"📊 Model Comparison")
print(f"Q: {comparison_question}")
print("=" * 60)

for model_name, model_id in GROQ_MODELS.items():
    print(f"\n🔷 {model_name}:")
    try:
        ans = ask_groq(comparison_question, SYSTEM_PROMPT, model=model_id)
        print(f"   {ans}")
    except Exception as e:
        print(f"   Error: {e}")

In [ ]:
# ── Interactive Q&A — ask your own questions ──────────────────────────────────

print("💬 Interactive Q&A with Groq LLaMA 3.3 70B")
print("Type 'exit' to stop\n")

while True:
    user_input = input("Your question: ").strip()
    if user_input.lower() in ["exit", "quit", ""]:
        print("Goodbye!")
        break
    print("\nAnswer:")
    answer = ask_groq(user_input, SYSTEM_PROMPT, model="llama-3.3-70b-versatile")
    print(answer)
    print("-" * 60)

---
## 📋 Summary & Quick Reference

| Component | Tool | Notes |
|---|---|---|
| **Base Model** | `TinyLlama-1.1B-Chat-v1.0` | 1.1B params, LLaMA architecture, fits T4 GPU |
| **Fine-Tuning Method** | LoRA (PEFT) | Only ~2M params trained (0.2% of total) |
| **Training Framework** | HuggingFace Trainer | Handles loop, logging, checkpointing |
| **Precision** | FP16 | Halves GPU memory usage |
| **GPU** | Colab T4 (15GB) | Free tier — sufficient for TinyLlama + LoRA |
| **Groq LLM** | LLaMA 3.3 70B | For concept explanation and Q&A |

### 🔑 Key Takeaways

1. **LoRA** makes fine-tuning accessible — trains 0.2% of parameters, saves 99.8% of VRAM
2. **TinyLlama** is 10x larger than GPT-2 but still fits on a free Colab GPU
3. **Fine-tuning** teaches the model your specific domain/style
4. **Groq API** is better when you need fast results without training
5. **HuggingFace Trainer** handles all training complexity automatically

### 📁 Files Created
- `/content/tinyllama-finetuned/` — full training checkpoints
- `/content/tinyllama-lora-adapter/` — just the LoRA weights (~10-20MB)